# Lake House

In [2]:
import pandas as pd
import numpy as np
from deltalake.writer import write_deltalake
import os
import boto3
from botocore.exceptions import ClientError
from pathlib import Path
# from minio import Minio
from dotenv import load_dotenv

load_dotenv('../../../.env_file/infra.env')

True

## Generate data

In [2]:
data_dict = {
    'date': [(pd.Timestamp.now() - pd.Timedelta(days=i)).date() for i in range(99, -1, -1)],
    'name': [''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz'), size=8)) for _ in range(100)],
    'price': np.random.random(100).tolist(),
    'quantity': np.random.randint(1, 1000, size=100).tolist(),
}

df = pd.DataFrame(data_dict)
df.head()

,date,name,price,quantity
0,2026-02-16,yjuyrfhh,0.396679,708
1,2026-02-17,wpnvfqno,0.628313,210
2,2026-02-18,ungsokrq,0.618191,330
3,2026-02-19,wxvdxdgq,0.130412,994
4,2026-02-20,tlirqntg,0.244901,469


In [3]:
split_idx = np.array_split(df.index, 3)
df_part_1, df_part_2, df_part_3 = [df.loc[idx].reset_index(drop=True) for idx in split_idx]
df_part_1, df_part_2, df_part_3

(          date      name     price  quantity
 0   2026-02-16  yjuyrfhh  0.396679       708
 1   2026-02-17  wpnvfqno  0.628313       210
 2   2026-02-18  ungsokrq  0.618191       330
 3   2026-02-19  wxvdxdgq  0.130412       994
 4   2026-02-20  tlirqntg  0.244901       469
 5   2026-02-21  fojdfruv  0.308868       882
 6   2026-02-22  jqrpusvk  0.985744       350
 7   2026-02-23  vakbsbxb  0.707473        62
 8   2026-02-24  fbbatisp  0.024849       395
 9   2026-02-25  yswmutnk  0.402232       209
 10  2026-02-26  btlfrcpg  0.938549       577
 11  2026-02-27  rqgspidk  0.139415       721
 12  2026-02-28  pxbliusb  0.220391       341
 13  2026-03-01  bbeprrek  0.221439       461
 14  2026-03-02  kteocksa  0.176381       299
 15  2026-03-03  cjmbszqw  0.679159       984
 16  2026-03-04  pkcmajfb  0.509960       785
 17  2026-03-05  yybzdjhr  0.870856       800
 18  2026-03-06  veehlmhf  0.389050       845
 19  2026-03-07  ixzrsyjz  0.352245        30
 20  2026-03-08  ozvnmoog  0.99397

In [ ]:
all_df:list[pd.DataFrame] = [df_part_1, df_part_2, df_part_3]
for idx, part in enumerate(all_df):
    write_deltalake(f'../../../data/test/lh/part_{idx}', part)

## Push data to S3

In [5]:
def create_bucket(bucket_name, region='ap-northeast-1'):
    """Create an S3 bucket in a specified region

    If a region is not specified, the bucket is created in the S3 default
    region (us-east-1).

    :param bucket_name: Bucket to create
    :param region: String region to create bucket in, e.g., 'us-west-2'
    :return: True if bucket created, else False
    """

    # Create bucket
    try:
        bucket_config = {}
        s3_client = boto3.client(
            's3',
            region_name=region,
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        if region != 'us-east-1':
            bucket_config['CreateBucketConfiguration'] = {'LocationConstraint': region}

        s3_client.create_bucket(Bucket=bucket_name, **bucket_config)
    except ClientError as e:
        print(e)
        return False
    return True

def upload_file(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    s3_client = boto3.client(
        's3',
        region_name='ap-northeast-1',
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
    )
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        print(e)
        return False
    return True

# create_bucket('flezi-agent', 'ap-northeast-1')
# upload_file('../../data/test/csv/04.customer.csv', 'flezi-test', 'nyc-taxi/04.customer.csv')

In [6]:
s3_client = boto3.client(
    's3',
    region_name='ap-northeast-1',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

bucket_name = 'flezi-data'
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Found bucker {bucket_name}. Skip creation')
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == '404':
        print(f'No bucket {bucket_name} found. Create a new one')
        create_bucket(bucket_name)

Found bucker flezi-data. Skip creation


In [ ]:
data_path = Path('../../data/test/lh')

for fp in list(data_path.rglob('*.parquet')) + list(data_path.rglob('*.json')):
    print(f"Uploading {fp}")
    upload_file(fp, bucket_name, fp._str[len('../../../data/test/lh/'):])

Uploading ../../data/test/lh/part_2/part-00000-d19a0e29-04ee-4390-905c-4a340e751569-c000.snappy.parquet
Uploading ../../data/test/lh/part_1/part-00000-b5b1f79e-1019-4c50-b882-3709672b3c07-c000.snappy.parquet
Uploading ../../data/test/lh/part_0/part-00000-25054456-fcd0-42d7-bbc8-9f72b8398694-c000.snappy.parquet
Uploading ../../data/test/lh/part_2/_delta_log/00000000000000000000.json
Uploading ../../data/test/lh/part_1/_delta_log/00000000000000000000.json
Uploading ../../data/test/lh/part_0/_delta_log/00000000000000000000.json


## Query from Trino

- Create schema:

    ```sql
    CREATE SCHEMA IF NOT EXISTS lakehouse.store
            WITH (location = 's3://flezi-data/');
    ```

- Register a schema for existing tables (repeat 3 times)

    ```sql
    CALL lakehouse.system.register_table(
        schema_name => 'store',
        table_name => 'product_0',
        table_location => 's3://flezi-data/part_0'
    )
    ```

    - By doing so, `Trino` automatically figures it out the suitable schema for these table and the corresponding log files $\to$ Can query from now on on `Trino`

- Apply a view for these tables for a better look:

    ```sql
    CREATE OR REPLACE VIEW lakehouse.store.product AS
        SELECT * FROM lakehouse.store.product_0
        UNION ALL
        SELECT * FROM lakehouse.store.product_1
        UNION ALL
        SELECT * FROM lakehouse.store.product_2;
    ```

- Ever since we use `DeltaLake`, every modification will be recorded so we can roll back easily

    ```sql
    INSERT INTO lakehouse.store.product_2 ("date", "name", "price", "quantity")
        VALUES (CURRENT_TIMESTAMP, 'dummy_product', 0.0, 0);
    ```

    - Make it 4 times

    - Then, new data + log file appearing in the bucket

- Roll back:
    - Đầu tiên, cần xác định version dữ liệu cần lấy ra thông qua lệnh

        ```bash
        use lakehouse.store;
        select version, timestamp, operation from "product_2$history";
        ```

        - Output:

            ```txt
            version  |          timestamp          | operation 
            ---------+-----------------------------+-----------
                   0 | 2026-04-21 03:49:08.480 UTC | WRITE     
                   1 | 2026-04-21 12:22:17.519 UTC | WRITE     
                   2 | 2026-04-21 12:22:23.375 UTC | WRITE     
                   3 | 2026-04-21 12:22:28.347 UTC | WRITE
            ```

    - Sau đó note lại version cần lấy và truy vấn theo 3 dạng sau:
        - TH1: Data đủ nhỏ để giữ trên RAM. Query trực tiếp `Trino` từ source $\to$ Trả về `Dataframe`
        - TH2: Data lớn hơn RAM. Lúc này cần lưu ra file parquet và dùng ML lib để train từ file đó. File output parquet có thể export ra local hoặc `MinIO`